# Naive to Advanced RAG — Bramblewood Gear Co. / FieldWise

**Company:** Bramblewood Gear Co., an outdoor equipment retailer (tents, sleeping bags,
backpacks, stoves, hiking footwear).

**Product:** FieldWise, an internal retrieval-augmented assistant that helps
support agents answer gear-care, troubleshooting, and warranty questions
without digging through several wikis by hand.

This notebook walks through FieldWise's retrieval pipeline from basic to more
advanced patterns, using Bramblewood's own support-knowledge-base articles:

- Basic vector-search RAG baseline
- Multi-query retrieval
- HyDE (hypothetical document embeddings)
- Parent-child retrieval
- Re-ranking

The motivating problem: a support agent's search phrasing rarely matches the
wording in the knowledge base ("my tent is leaking overnight" vs. an article
about "rainfly waterproofing failure and seam sealing"). Each section below
adds one technique to close that gap.

## Learning goals

By the end of this notebook, you should be able to:

1. Build a naive vector-search-based RAG baseline.
2. Expand a query into multiple variants.
3. Use a HyDE-style hypothetical answer to improve retrieval.
4. Retrieve small chunks while keeping parent documents intact.
5. Add a reranking pass on top of the initial retrieval results.

## 1) Install packages

```bash
pip install -U scikit-learn langchain langchain-community langchain-text-splitters langchain-core faiss-cpu
```

This notebook's default embedding path is a local, offline-safe embedding
(see Section 4). If you want to try the live semantic embedding model
instead, also install `sentence-transformers` and `langchain-huggingface`.

In [ ]:
%pip install -qU scikit-learn langchain langchain-community langchain-text-splitters langchain-core faiss-cpu

## 2) Bramblewood's support knowledge base

A handful of real support-knowledge-base articles: gear care, returns and
warranty policy, and a couple of notes on how FieldWise itself is organized.

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=(
            "The Summit Ridge 2P Tent and Alpine Trail 4P Tent both ship with a "
            "factory-applied waterproof rainfly coating that wears down after "
            "repeated UV exposure. Once the coating wears down, seams and fabric "
            "panels can start to seep during heavy rain. Reapply a DWR (durable "
            "water repellent) treatment once a year and reseal the floor and "
            "rainfly seams with seam sealer to restore waterproofing."
        ),
        metadata={"source": "tent_care_waterproofing", "topic": "tents"},
    ),
    Document(
        page_content=(
            "Bramblewood accepts returns within 60 days for unused gear still in "
            "its original packaging. Used or worn gear is not eligible for a "
            "general return; it can only be exchanged or repaired through a "
            "warranty defect claim. Hard goods such as tents and the TrailFlame "
            "stove carry a 2-year warranty against manufacturing defects, and "
            "Basecamp backpacks carry a lifetime warranty against stitching and "
            "workmanship defects."
        ),
        metadata={"source": "returns_and_warranty_policy", "topic": "policy"},
    ),
    Document(
        page_content=(
            "FieldWise, Bramblewood's internal support assistant, indexes "
            "gear-care articles, warranty terms, and troubleshooting notes into "
            "a shared knowledge base so agents can pull a grounded answer "
            "instead of searching several wikis by hand during a live chat."
        ),
        metadata={"source": "fieldwise_overview", "topic": "internal"},
    ),
    Document(
        page_content=(
            "A good support knowledge base returns the passage that is most "
            "relevant to the agent's question and keeps the final answer "
            "limited to what that passage actually supports, rather than "
            "guessing at gear specifications that were never documented."
        ),
        metadata={"source": "support_best_practices", "topic": "rag"},
    ),
]

documents

## 3) Split into chunks

Breaking longer articles into smaller chunks keeps each retrieved passage
focused, which matters once several products share one article.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=140,
    chunk_overlap=20,
)

chunks = splitter.split_documents(documents)

print("Chunk count:", len(chunks))
for i, chunk in enumerate(chunks, 1):
    print(f"--- Chunk {i} ---")
    print(chunk.page_content)
    print(chunk.metadata)

## 4) Embedding availability check

FieldWise's production deployment uses a local Hugging Face
sentence-transformer for embeddings. That model has to be downloaded the
first time it runs, so any environment without network access to Hugging
Face (a locked-down build server, an air-gapped office, this notebook's own
sandbox) needs a fallback that keeps the exact same `Embeddings` interface.
We check for the live model once, up front, and fall back to a deterministic
offline embedding if it isn't reachable — everything downstream (FAISS,
retrievers) is written against the interface, not the specific
implementation, so it doesn't need to know which one is active.

In [ ]:
from langchain_core.embeddings import Embeddings
from sklearn.feature_extraction.text import HashingVectorizer


class OfflineHashingEmbeddings(Embeddings):
    """Deterministic, dependency-free embedding fallback.

    Uses scikit-learn's hashing trick, so it needs no fitted vocabulary and
    produces the same vector for the same text regardless of which process
    or notebook session calls it. It is a lexical fallback, not a semantic
    one -- it won't know "leak" and "waterproofing failure" are related the
    way a real sentence-transformer would. That's the trade-off for an
    offline-safe default, and it's exactly why query rewriting, multi-query
    retrieval, and HyDE (this notebook's next three sections) earn their
    keep: they close the wording gap that a purely lexical embedding can't.
    """

    def __init__(self, n_features: int = 512):
        self.n_features = n_features
        self._hv = HashingVectorizer(
            n_features=n_features, alternate_sign=False, norm="l2", stop_words="english"
        )

    def embed_documents(self, texts):
        return self._hv.transform(texts).toarray().tolist()

    def embed_query(self, text):
        return self._hv.transform([text]).toarray()[0].tolist()


def get_embeddings():
    """Try the live sentence-transformers model; fall back to the offline
    hashing embeddings if it can't be loaded (package missing, no network
    to huggingface.co, or any other environment issue)."""
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
        candidate = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        candidate.embed_query("connectivity check")
        print("Embeddings: live sentence-transformers model loaded (semantic).")
        return candidate, "live"
    except Exception as e:
        print(f"Embeddings: falling back to offline hashing embeddings -- {type(e).__name__}: {str(e)[:160]}")
        return OfflineHashingEmbeddings(), "offline"


embeddings, EMBEDDINGS_MODE = get_embeddings()

## 5) Build a simple vector store

FAISS gives us a naive retriever baseline before trying more advanced
strategies.

In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print("Vector store ready. Embeddings mode:", EMBEDDINGS_MODE)

## 6) Naive RAG baseline

The naive version of RAG is straightforward:

1. Embed the question.
2. Retrieve the closest chunks.
3. Feed those chunks into the answer step later.

Here's the query an agent might actually type into FieldWise after reading a
ticket.

In [ ]:
query = "customer says their tent is leaking overnight, what should I tell them?"
docs = retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"--- Retrieved {i} ---")
    print(doc.page_content)
    print(doc.metadata)

## 7) Basic RAG context builder

The simplest version of "use retrieval to answer": retrieve, join the
chunks, and pass that text along to a model prompt later.

In [ ]:
def build_context(query: str, k: int = 3) -> str:
    docs = vector_store.similarity_search(query, k=k)
    return "\n".join(
        f"[source={doc.metadata.get('source')}]: {doc.page_content}"
        for doc in docs
    )

context = build_context(query)
print(context)

## 8) Multi-query retrieval

The agent's ticket-derived query uses words like "leaking" and "overnight."
The knowledge-base article uses "seep," "waterproofing," and "seams." A
basic multi-query approach expands the question into a few variants that
bridge that gap and merges the results.

In [ ]:
def expand_query(query: str) -> list[str]:
    return [
        query,
        query.replace("leaking", "seeping through the seams and rainfly"),
        query.replace("overnight", "during heavy rain"),
    ]

expanded_queries = expand_query(query)
expanded_queries

In [ ]:
def multi_query_retrieve(query: str, k: int = 2):
    seen = set()
    merged = []
    for q in expand_query(query):
        for doc in vector_store.similarity_search(q, k=k):
            key = (doc.metadata.get("source"), doc.page_content)
            if key not in seen:
                seen.add(key)
                merged.append(doc)
    return merged

mq_docs = multi_query_retrieve(query)
for i, doc in enumerate(mq_docs, 1):
    print(f"--- Multi-query {i} ---")
    print(doc.page_content)
    print(doc.metadata)

## 9) HyDE

HyDE means hypothetical document embeddings: write a plausible answer first,
then embed *that* text and retrieve with it, instead of embedding the raw
(often underspecified) question.

In [ ]:
def hyde_hypothesis(query: str) -> str:
    return (
        f"A helpful answer to '{query}' would likely mention reapplying a DWR "
        "(durable water repellent) treatment to the rainfly, resealing the "
        "seams with seam sealer, and explaining whether the issue is covered "
        "under the hard-goods warranty."
    )

hyde_text = hyde_hypothesis(query)
print(hyde_text)

hyde_docs = vector_store.similarity_search(hyde_text, k=3)
for i, doc in enumerate(hyde_docs, 1):
    print(f"--- HyDE {i} ---")
    print(doc.page_content)
    print(doc.metadata)

## 10) Parent-child retrieval

This pattern keeps small chunks for matching, but returns the larger parent
article so the agent doesn't lose surrounding context (for example, the
warranty caveat that sits right after the care instructions).

1. Split parent articles into child chunks.
2. Retrieve the child chunks.
3. Map each child back to its parent.
4. Return the parent documents.

In [ ]:
parent_documents = [
    Document(
        page_content=(
            "Parent A -- Tent leak troubleshooting: check the rainfly coating "
            "first, since most reported leaks come from worn-off DWR treatment "
            "rather than a manufacturing defect. Reseal exposed seams with seam "
            "sealer. If leaking continues after reproofing and the tent is under "
            "two years old, file it as a hard-goods warranty claim instead of a "
            "general return."
        ),
        metadata={"parent_id": "A"},
    ),
    Document(
        page_content=(
            "Parent B -- Returns vs. warranty: a 60-day return only applies to "
            "unused gear in original packaging. Anything that has been used "
            "outdoors, including a tent that has already been pitched, is "
            "handled as a warranty claim if the issue looks like a defect, not "
            "as a standard return."
        ),
        metadata={"parent_id": "B"},
    ),
]

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=90, chunk_overlap=10)
child_chunks = parent_splitter.split_documents(parent_documents)

child_to_parent = {p.metadata["parent_id"]: p for p in parent_documents}

child_store = FAISS.from_documents(child_chunks, embeddings)
child_retriever = child_store.as_retriever(search_kwargs={"k": 2})

child_hits = child_retriever.invoke("is a leaking tent covered or is it just a return?")
parent_hits = []
seen_parent_ids = set()

for child in child_hits:
    pid = child.metadata.get("parent_id")
    if pid and pid not in seen_parent_ids:
        seen_parent_ids.add(pid)
        parent_hits.append(child_to_parent[pid])

for i, doc in enumerate(parent_hits, 1):
    print(f"--- Parent {i} ---")
    print(doc.page_content)
    print(doc.metadata)

## 11) Re-ranking

Reranking usually means: retrieve a larger candidate set first, then sort
those candidates again with a more precise scoring pass.

In [ ]:
candidate_docs = vector_store.similarity_search(query, k=4)

def simple_rerank(query: str, docs):
    stop_terms = {
        "a", "an", "the", "is", "are", "was", "were", "what", "should",
        "i", "we", "do", "does", "did", "how", "why", "who", "to", "of",
        "in", "on", "for", "and", "or", "them", "their", "it", "its", "tell",
    }
    q_terms = {t for t in query.lower().split() if t not in stop_terms}
    scored = []
    for doc in docs:
        text = doc.page_content.lower()
        score = sum(1 for term in q_terms if term in text)
        scored.append((score, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for score, doc in scored]

reranked = simple_rerank(query, candidate_docs)
for i, doc in enumerate(reranked, 1):
    print(f"--- Reranked {i} ---")
    print(doc.page_content)
    print(doc.metadata)

## 12) Optional real cross-encoder setup

If you want a stronger reranking pass on a live machine, the LangChain
reranker docs show a `HuggingFaceCrossEncoder` with `CrossEncoderReranker`
and `ContextualCompressionRetriever`. A common local choice is a small BGE
reranker model such as `BAAI/bge-reranker-v2-m3`. This needs network access
to Hugging Face the first time it runs, so it's left commented out here.

In [ ]:
# Optional advanced setup (needs live network access to Hugging Face) --
# keep commented out until you want to run it on your own machine.
# from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
# from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
# from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
# reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)
# compression_retriever = ContextualCompressionRetriever(
#     base_compressor=reranker,
#     base_retriever=vector_store.as_retriever(search_kwargs={"k": 10}),
# )
# docs = compression_retriever.invoke(query)
# docs

print("Optional reranker template ready.")

## 13) Compare the strategies

- Naive RAG: one query, one retrieval step
- Multi-query: one question, several query variants
- HyDE: retrieve from a hypothetical answer embedding
- Parent-child: match on small chunks, return larger context
- Rerank: refine the shortlist with a stronger scoring pass

In [ ]:
summary = {
    "naive": len(retriever.invoke(query)),
    "multi_query": len(mq_docs),
    "hyde": len(hyde_docs),
    "parent_child": len(parent_hits),
    "reranked_candidates": len(reranked),
}
summary

## 14) Key takeaways

- Start with a naive retriever baseline.
- Query expansion recovers passages the agent's wording missed.
- HyDE helps when the ticket-derived query is short or informally worded.
- Parent-child retrieval preserves the warranty caveat that sits next to the
  care instructions, instead of losing it to chunk boundaries.
- Reranking is often the biggest quality boost after the initial retrieval
  step, and it's cheap to add on top of anything above.
- The offline embedding fallback in Section 4 is deliberately lexical, which
  makes the wording gap in Section 6 more visible than it would be with a
  full semantic model -- worth re-running this notebook with a live
  `sentence-transformers` model installed to see the difference.

## References

- Retrieval: https://docs.langchain.com/oss/python/langchain/retrieval
- Semantic search / knowledge base: https://docs.langchain.com/oss/python/langchain/knowledge-base
- Retriever integrations: https://docs.langchain.com/oss/python/integrations/retrievers/index
- Cross encoder reranker: https://docs.langchain.com/oss/python/integrations/document_transformers/cross_encoder_reranker

## Verification summary

| Section | Status | Notes |
|---|---|---|
| Package install | Executed | pip install cell |
| Knowledge base + chunking | Executed & verified | in-memory documents, deterministic |
| Embeddings | Executed & verified | see printed `EMBEDDINGS_MODE` above -- offline hashing fallback in a sandbox with no Hugging Face access, live sentence-transformers on a machine with internet |
| Naive / multi-query / HyDE / parent-child / rerank | Executed & verified | pure Python + vector search, no external API calls |
| Optional cross-encoder reranker | Written, not executed | needs live Hugging Face access; left commented out |

This notebook has no Groq or LangSmith dependency, so everything above runs
identically whether or not those services are reachable.